# Differential labels: is the cancer specific signal learnable?

## Why this run

The raw label run predicts essentiality at r around 0.54. Most of that is generic.
Those are genes essential in every cancer, not in kidney in particular. The previous
run showed this signal is largely connectivity, and connectivity is the same across
cancers.

So this run removes the generic part and asks what is left.

## The one change

For every gene we take its mean CRISPR score across all DepMap cell lines, and subtract
it from the kidney score. What remains is essentiality relative to other cancers.

The graph, the features, the architecture, the folds and the epochs are all the same as
the raw label run. Only the label changed.

## Two checks before trusting the result

The global means are printed for HIF1A and PAX8. If a gene's pan cancer mean is near
zero there is nothing to subtract, so it cannot show a kidney specific shift.

Label spread is measured at the end. A near zero r only means something if the labels
themselves are not flat.

## Degree

Raw essentiality tracks how connected a gene is, because hubs are broadly essential.
Subtracting a per gene constant should remove that. The correlation is computed both
ways to check.

## What to look at

The Pearson r, then the per gene attention. These two do not have to move in the same
direction.


In [ ]:
!pip install torch-geometric


In [ ]:
import torch
print(torch.__version__)
print(torch.version.cuda)

In [ ]:
!pip install torch-geometric



In [ ]:
import pandas as pd
import numpy as np
import torch
from torch_geometric.data import Data
from scipy.stats import pearsonr
from scipy.stats import pearsonr

import random
from torch_geometric.loader import DataLoader


In [ ]:
# same inputs as the raw-label run, plus the full DepMap matrix, which is
# only needed to build the pan-cancer mean below
ppi = pd.read_csv("/content/humanbase_edges.csv")
labels = pd.read_csv("/content/labels.csv")
expressions = pd.read_csv("/content/expression.csv")
mutations = pd.read_csv("/content/mutations.csv")
raw_label = pd.read_csv("/content/CRISPRGeneEffect.csv")

In [ ]:
raw_label.rename(columns={"Unnamed: 0": "ModelID"}, inplace=True)
# per-gene mean over all DepMap lines. the 32 kidney lines are inside it (~3%)
global_mean = raw_label.drop("ModelID", axis=1).mean(axis=0)
# "SYMBOL (Entrez)" -> "SYMBOL", to match the network's gene names
global_mean.index = global_mean.index.str.split(" (", regex=False).str[0]
global_mean

In [ ]:
ppi.shape

In [ ]:
# node index comes from the network, not from DepMap
ppi_genes = pd.concat([ppi['symbol_a'], ppi['symbol_b']]).unique()
gene_to_idx = {gene: i for i, gene in enumerate(ppi_genes)}

In [ ]:
edge_src = ppi['symbol_a'].map(gene_to_idx).values
edge_dst = ppi['symbol_b'].map(gene_to_idx).values

# symmetrised, so messages pass both ways
edge_index = torch.tensor(
    np.array([np.concatenate([edge_src, edge_dst]),
              np.concatenate([edge_dst, edge_src])]),
    dtype=torch.long
)


# posterior, raw. duplicated to match the doubled edge_index
edge_features = torch.tensor(ppi[['posterior']].values, dtype=torch.float)
edge_attr = torch.cat([edge_features, edge_features], dim=0)

In [ ]:
expr_no_id = expressions.drop('ModelID', axis=1)
mut_no_id = mutations.drop('ModelID', axis=1)
# genes missing from the mutation file become 0
mutations_aligned = mut_no_id.reindex(columns=ppi_genes, fill_value=0)
labels_no_id = labels.drop('ModelID', axis=1)
labels_reindexed = labels_no_id.reindex(columns=ppi_genes)
# the only change from the raw-label run: subtract the pan-cancer mean, so
# what remains is essentiality relative to other cancers rather than absolute
labels_reindexed = labels_reindexed - global_mean.reindex(labels_reindexed.columns)

In [ ]:
# check the two means are usable before trusting anything downstream.
# a gene whose global mean is near zero has nothing to subtract.
print(f"HIF1A global mean: {global_mean['HIF1A']:.4f}")
print(f"PAX8 global mean: {global_mean['PAX8']:.4f}")

In [ ]:
from scipy.stats import spearmanr

# does either label track node degree? raw essentiality should, since hubs are
# broadly essential. subtracting a per-gene constant should remove it.
deg = pd.concat([ppi['symbol_a'], ppi['symbol_b']]).value_counts()

lab = labels.drop('ModelID', axis=1)
raw_mean  = lab.mean(axis=0)
diff_mean = (lab - global_mean.reindex(lab.columns)).mean(axis=0)

g = deg.index.intersection(raw_mean.dropna().index).intersection(diff_mean.dropna().index)
print("genes compared:", len(g))
print("raw  vs degree:", spearmanr(deg[g], raw_mean[g]))
print("diff vs degree:", spearmanr(deg[g], diff_mean[g]))

In [ ]:
# one graph per cell line. topology and edge_attr are shared, only x differs
graph_list = []
for i in range(32):
    expr_row = expr_no_id[ppi_genes].iloc[i].values
    mut_row = mutations_aligned.iloc[i].values
    x = torch.tensor(np.column_stack([expr_row, mut_row]), dtype=torch.float)

    y_raw = labels_reindexed.iloc[i]
    # unlabelled genes filled with 0 for shape, then masked out of the loss
    cell_mask = torch.tensor((~y_raw.isna().values) & np.isin(ppi_genes, labels_no_id.columns), dtype=torch.bool)
    y = torch.tensor(y_raw.fillna(0).values, dtype=torch.float)

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y, mask=cell_mask)
    graph_list.append(data)

In [ ]:
import torch.nn as nn
from torch_geometric.nn import GATConv

In [ ]:
# identical to the raw-label run. only the label changed.
class KidneyGAT(nn.Module):
    def __init__(self):
        super().__init__()
        # one encoder per feature, 32 + 32 -> 64
        self.expr_encoder = nn.Linear(1, 32)
        self.mut_encoder = nn.Linear(1, 32)
        # no fill_value, so PyG fills self-loops with the mean posterior
        self.conv1 = GATConv(64, 64, heads=4, edge_dim=1)
        # heads concat in layer 1 (64*4 in), average in layer 2
        self.conv2 = GATConv(64 * 4, 64, heads=4, edge_dim=1, concat=False)
        self.relu = nn.LeakyReLU()
        self.dropout = nn.Dropout(0.3)
        self.out = nn.Linear(64, 1)

    def forward(self, data):
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr

        expr = self.expr_encoder(x[:, 0:1])
        mut = self.mut_encoder(x[:, 1:2])
        x = torch.cat([expr, mut], dim=1)

        x = self.conv1(x, edge_index, edge_attr)
        x = self.relu(x)
        x = self.dropout(x)

        x = self.conv2(x, edge_index, edge_attr)
        x = self.relu(x)
        x = self.dropout(x)

        # one score per gene
        x = self.out(x)
        return x.squeeze()

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = KidneyGAT().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
loss_fn = nn.MSELoss()

In [ ]:
from sklearn.model_selection import KFold

# folds split cell lines, not genes
kf = KFold(n_splits=5, shuffle=True, random_state=12)
all_attn = []
fold_pearsons = []

for fold, (train_idx, test_idx) in enumerate(kf.split(graph_list)):
    train_data = [graph_list[i] for i in train_idx]
    test_data = [graph_list[i] for i in test_idx]
    train_loader = DataLoader(train_data, batch_size=1, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=1)

    # Fresh model each fold
    model = KidneyGAT().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
    loss_fn = nn.MSELoss()

    # Train
    for epoch in range(200):
        model.train()
        for batch in train_loader:
            batch = batch.to(device)
            pred = model(batch)
            loss = loss_fn(pred[batch.mask], batch.y[batch.mask])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # Evaluate + extract attention
    model.eval()
    fold_corrs = []
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)

            # Pearson
            # correlation across genes, within one held-out cell line
            pred = model(batch)
            p = pred[batch.mask].cpu().numpy()
            a = batch.y[batch.mask].cpu().numpy()
            r, _ = pearsonr(p, a)
            fold_corrs.append(r)

            # Attention extraction
            # forward pass repeated to reach the attention weights.
            # conv1's are discarded, only conv2's are kept.
            expr = model.expr_encoder(batch.x[:, 0:1])
            mut = model.mut_encoder(batch.x[:, 1:2])
            x = torch.cat([expr, mut], dim=1)
            x, _ = model.conv1(x, batch.edge_index, batch.edge_attr, return_attention_weights=True)
            x = model.relu(x)
            x = model.dropout(x)
            x, (edge_idx, scores) = model.conv2(x, batch.edge_index, batch.edge_attr, return_attention_weights=True)

            # average over the 4 heads
            avg_scores = scores.mean(dim=1).cpu().numpy()
            all_attn.append(avg_scores)

    fold_r = np.mean(fold_corrs)
    fold_pearsons.append(fold_r)
    print(f"Fold {fold+1}: Pearson r = {fold_r:.4f}")

print(f"\nMean Pearson r across 5 folds: {np.mean(fold_pearsons):.4f} ± {np.std(fold_pearsons):.4f}")

In [ ]:
# Get edge indices from one sample
# this pass is only for edge_idx, identical across graphs. `model` here is
# the last fold's.
sample = graph_list[0].to(device)
with torch.no_grad():
    expr = model.expr_encoder(sample.x[:, 0:1])
    mut = model.mut_encoder(sample.x[:, 1:2])
    x = torch.cat([expr, mut], dim=1)
    x, _ = model.conv1(x, sample.edge_index, sample.edge_attr, return_attention_weights=True)
    x = model.relu(x)
    x = model.dropout(x)
    x, (edge_idx, _) = model.conv2(x, sample.edge_index, sample.edge_attr, return_attention_weights=True)

src = edge_idx[0].cpu().numpy()
dst = edge_idx[1].cpu().numpy()

# Average attention across all folds/cell lines
# 32 arrays, one per held-out cell line
mean_attn = np.mean(all_attn, axis=0)

# Filter self-loops
# PyG appended one per node
mask = src != dst
src_filtered = src[mask]
dst_filtered = dst[mask]
attn_filtered = mean_attn[mask]

idx_to_gene = {i: gene for gene, i in gene_to_idx.items()}

# Top 100 interactions
# unconditional ranking, not degree-normalised
top_indices = np.argsort(attn_filtered)[-100:][::-1]
print("Top 100 gene interactions (averaged across all folds):\n")
for rank, i in enumerate(top_indices, 1):
    g1 = idx_to_gene[src_filtered[i]]
    g2 = idx_to_gene[dst_filtered[i]]
    print(f"{rank:2d}. {g1:10s} <-> {g2:10s}  attn: {attn_filtered[i]:.4f}")

In [ ]:
# per-gene view: conditioned on one gene, where does its attention go
known_cancer_genes = ['VHL', 'TP53', 'EGFR', 'PTEN', 'MTOR', 'MYC', 'KRAS', 'BRCA1', 'PIK3CA', 'BRAF']

for gene in known_cancer_genes:
    if gene not in gene_to_idx:
        print(f"{gene} — not in PPI network")
        continue

    idx = gene_to_idx[gene]
    # either direction, since the graph was symmetrised
    gene_mask = (src_filtered == idx) | (dst_filtered == idx)
    gene_attn = attn_filtered[gene_mask]
    gene_src = src_filtered[gene_mask]
    gene_dst = dst_filtered[gene_mask]

    if len(gene_attn) == 0:
        print(f"{gene} — no edges found")
        continue

    top5 = np.argsort(gene_attn)[-5:][::-1]

    print(f"\n{gene} — {len(gene_attn)} edges, max attn: {gene_attn.max():.4f}")
    for i in top5:
        partner = idx_to_gene[gene_dst[i]] if gene_src[i] == idx else idx_to_gene[gene_src[i]]
        print(f"  <-> {partner:12s}  attn: {gene_attn[i]:.4f}")

In [ ]:
# are the labels degenerate? a near-zero r is only interpretable if the
# labels themselves have spread.
vals = labels_reindexed.values.flatten()
vals = vals[~np.isnan(vals)]
print("count:", len(vals))
print("std:", np.std(vals))
print("% within ±0.1:", round(np.mean(np.abs(vals) < 0.1) * 100, 1))
print("min/max:", np.min(vals), np.max(vals))